In [ ]:
!pip install --upgrade sympy torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 150.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200

In [31]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import copy
import time
import os
import torch.nn.functional as F


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
print(f"Using device: {DEVICE}")

# ││ Data loading

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)


#

def evaluate(model, loader, device=DEVICE):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total


def count_nonzero(model):
    """Count non-zero params using effective (masked) weights."""
    total = 0
    nonzero = 0
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w = module.weight
            total += w.numel()
            nonzero += (w != 0).sum().item()
    return total, nonzero


def fine_tune(model, epochs, lr=0.0001):
    """Fine-tune a pruned model."""
    if epochs == 0:
        return model
    model.train()
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                          weight_decay=0.0005)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=lr * 0.01
    )

    for epoch in range(epochs):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            logits = model(inputs)
            loss = nn.CrossEntropyLoss()(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()

    return model


def load_fresh_model(model_path):
    """Load a fresh copy of the trained model."""
    model = VGGStyleCNN().to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=False))
    return model

Using device: cuda


In [23]:
# @title
class VGGStyleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # ── Block 1: 3 → 64 → 64, pool ──
        self.conv1a = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1a = nn.BatchNorm2d(64)
        self.conv1b = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn1b = nn.BatchNorm2d(64)

        # ── Block 2: 64 → 128 → 128, pool ──
        self.conv2a = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2a = nn.BatchNorm2d(128)
        self.conv2b = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn2b = nn.BatchNorm2d(128)

        # ── Block 3: 128 → 256 → 256, pool ──
        self.conv3a = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3a = nn.BatchNorm2d(256)
        self.conv3b = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn3b = nn.BatchNorm2d(256)

        # ── Block 4: 256 → 512 → 512, pool ──
        self.conv4a = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn4a = nn.BatchNorm2d(512)
        self.conv4b = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.bn4b = nn.BatchNorm2d(512)

        # ── Classifier ──
        self.fc1 = nn.Linear(512, 256)
        self.fc2 = nn.Linear(256, num_classes)

        # Initialize weights (He initialization, same as your original)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)   # gamma = 1
                nn.init.zeros_(m.bias)    # beta = 0
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # Block 1: (N,3,32,32) → (N,64,16,16)
        x = F.relu(self.bn1a(self.conv1a(x)))
        x = F.relu(self.bn1b(self.conv1b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.1, training=self.training)

        # Block 2: (N,64,16,16) → (N,128,8,8)
        x = F.relu(self.bn2a(self.conv2a(x)))
        x = F.relu(self.bn2b(self.conv2b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.2, training=self.training)

        # Block 3: (N,128,8,8) → (N,256,4,4)
        x = F.relu(self.bn3a(self.conv3a(x)))
        x = F.relu(self.bn3b(self.conv3b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.3, training=self.training)

        # Block 4: (N,256,4,4) → (N,512,2,2)
        x = F.relu(self.bn4a(self.conv4a(x)))
        x = F.relu(self.bn4b(self.conv4b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.4, training=self.training)

        # GAP: (N,512,2,2) → (N,512)
        x = F.adaptive_avg_pool2d(x, 1)
        x = x.view(x.size(0), -1)

        # Classifier
        x = F.relu(self.fc1(x))
        x = F.dropout(x, 0.5, training=self.training)
        x = self.fc2(x)          # raw logits — use nn.CrossEntropyLoss

        return x



In [24]:


#Global vs. Layer-wise Pruning






def apply_global_pruning(model, sparsity):
    """Prune globally score all weights and remove smallest."""


    params_to_prune = []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            params_to_prune.append((module, 'weight'))
    prune.global_unstructured(
        params_to_prune, pruning_method=prune.L1Unstructured, amount=sparsity
    )
    return model






def apply_layerwise_pruning(model, sparsity):
    """Prune layer independ but same sparsity level."""



    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            prune.l1_unstructured(module, name='weight', amount=sparsity)
    return model







def run_aspect1_global_vs_layerwise(model_path, fine_tune_epochs=5):
    """
    Compare global vs. layer-wise pruning across multiple sparsity levels.
    """
    print("\n" + "=" * 65)
    print("Global vs. Layer-wise Pruning")
    print("=" * 65)

    sparsity_levels = [0.3, 0.5, 0.7, 0.8, 0.9, 0.95]
    results_global = []
    results_layerwise = []

    for sparsity in sparsity_levels:
        print(f"\n  --- Sparsity: {sparsity*100:.0f}% ---")

        # Global pruning
        model_g = load_fresh_model(model_path)
        model_g = apply_global_pruning(model_g, sparsity)
        acc_g_before = evaluate(model_g, test_loader)
        _, nonzero_g = count_nonzero(model_g)
        model_g = fine_tune(model_g, fine_tune_epochs)
        acc_g_after = evaluate(model_g, test_loader)
        print(f"    Global:    {acc_g_before:.2f}% → {acc_g_after:.2f}% "
              f"(non-zero: {nonzero_g:,})")

        results_global.append({
            'sparsity': sparsity, 'acc_before': acc_g_before,
            'acc_after': acc_g_after, 'nonzero': nonzero_g
        })

        # Layer-wise pruning
        model_l = load_fresh_model(model_path)
        model_l = apply_layerwise_pruning(model_l, sparsity)
        acc_l_before = evaluate(model_l, test_loader)
        _, nonzero_l = count_nonzero(model_l)
        model_l = fine_tune(model_l, fine_tune_epochs)
        acc_l_after = evaluate(model_l, test_loader)
        print(f"    Layerwise: {acc_l_before:.2f}% → {acc_l_after:.2f}% "
              f"(non-zero: {nonzero_l:,})")

        results_layerwise.append({
            'sparsity': sparsity, 'acc_before': acc_l_before,
            'acc_after': acc_l_after, 'nonzero': nonzero_l
        })

    return results_global, results_layerwise


def plot_aspect1(results_global, results_layerwise):
    os.makedirs("plots", exist_ok=True)






    sparsities = [r['sparsity'] * 100 for r in results_global]
    acc_global = [r['acc_after'] for r in results_global]
    acc_layer = [r['acc_after'] for r in results_layerwise]
    acc_global_before = [r['acc_before'] for r in results_global]
    acc_layer_before = [r['acc_before'] for r in results_layerwise]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))





    # After fine-tuning comparison
    ax1.plot(sparsities, acc_global, 'bo-', linewidth=2, markersize=8,
             label='Global pruning')
    ax1.plot(sparsities, acc_layer, 'rs-', linewidth=2, markersize=8,
             label='Layer-wise pruning')
    ax1.set_xlabel('Sparsity (%)', fontsize=12)
    ax1.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax1.set_title('After Fine-tuning', fontsize=13)
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)








    # Before fine-tuning comparison
    ax2.plot(sparsities, acc_global_before, 'bo--', linewidth=2, markersize=8,
             label='Global pruning')
    ax2.plot(sparsities, acc_layer_before, 'rs--', linewidth=2, markersize=8,
             label='Layer-wise pruning')
    ax2.set_xlabel('Sparsity (%)', fontsize=12)
    ax2.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax2.set_title('Before Fine-tuning', fontsize=13)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    fig.suptitle('Ablation: Global vs. Layer-wise Pruning', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig("plots/ablation_global_vs_layerwise.png", dpi=150,
                bbox_inches='tight')
    plt.close()
    print("Saved: plots/ablation_global_vs_layerwise.png")


In [26]:


# Effect of Fine-tuning Epochs

def fine_tune_with_tracking(model, max_epochs, lr=0.0001):
    """
    Fine-tune and record test accuracy AFTER EACH epoch.
    This gives us a smooth recovery curve instead of isolated points.
    """
    if max_epochs == 0:
        return model, []

    model.train()
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                          weight_decay=0.0005)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max_epochs, eta_min=lr * 0.01
    )

    epoch_accs = []
    for epoch in range(max_epochs):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            logits = model(inputs)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()

        acc = evaluate(model, test_loader)
        epoch_accs.append(acc)
        print(f"    Epoch {epoch+1}/{max_epochs} | Test Acc: {acc:.2f}%")

    return model, epoch_accs










def run_aspect2_finetune_epochs(model_path, sparsity=0.95):
    """
    Run fine-tuning recovery at 90% and 95% and 99 sparsity.

    """
    print("\n" + "=" * 65)
    print("  ASPECT 2: Fine-tuning Epochs (90%, 95% and 99% sparsity)")
    print("=" * 65)

    max_epochs = 30
    all_results = {}

    for s in [0.90, 0.95, 0.99]:
        print(f"\n  ── Sparsity: {s*100:.0f}% ──")
        model = load_fresh_model(model_path)
        model = apply_global_pruning(model, s)

        acc_before = evaluate(model, test_loader)
        print(f"  Accuracy after pruning: {acc_before:.2f}%")
        print(f"  Fine-tuning for {max_epochs} epochs...\n")

        model, epoch_accs = fine_tune_with_tracking(model, max_epochs)

        # Build results: epoch 0 = no fine-tuning, then 1..max_epochs
        results = [{'epochs': 0, 'acc_before': acc_before, 'acc_after': acc_before}]
        for i, acc in enumerate(epoch_accs):
            results.append({
                'epochs': i + 1,
                'acc_before': acc_before,
                'acc_after': acc,
            })
        all_results[s] = results

    return all_results
















def plot_aspect2(all_results, sparsity=0.95):
    """Plot both 90% and 95% recovery curves on the same figure."""
    os.makedirs("plots", exist_ok=True)

    fig, ax = plt.subplots(figsize=(10, 6))

    colors = {'0.9': '#2196F3', '0.95': '#E91E63', '0.99': '#FF9800'}
    labels = {'0.9': '90% sparsity', '0.95': '95% sparsity', '0.99': '99% sparsity'}

    for s in [0.9, 0.95, 0.99]:
        results = all_results[s]
        epochs = [r['epochs'] for r in results]
        accs = [r['acc_after'] for r in results]
        acc_before = results[0]['acc_before']
        c = colors[str(s)]
        lab = labels[str(s)]

        # Recovery curve
        ax.plot(epochs, accs, 'o-', color=c, linewidth=2, markersize=6,
                label=f'{lab} (start: {acc_before:.1f}%)')








        # Desh line
        ax.axhline(y=acc_before, color=c, linestyle='--', linewidth=1, alpha=0.4)








        # shade region
        ax.fill_between(epochs, acc_before, accs, alpha=0.08, color=c)








        # start and end
        ax.annotate(f'{acc_before:.1f}%', (0, acc_before),
                    textcoords="offset points", xytext=(-30, -12 if s == 0.95 else 10),
                    ha='center', fontsize=9, color=c, fontweight='bold')
        ax.annotate(f'{accs[-1]:.1f}%', (epochs[-1], accs[-1]),
                    textcoords="offset points", xytext=(18, -5 if s == 0.9 else 5),
                    ha='center', fontsize=9, color=c, fontweight='bold')











    # Baseline reference
    ax.axhline(y=90.68, color='gray', linestyle=':', linewidth=1.5,
               label='Unpruned baseline (90.68%)', alpha=0.5)

    ax.set_xlabel('Fine-tuning Epochs', fontsize=12)
    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.set_title('Accuracy Recovery During Fine-tuning at Different Sparsity Levels',
                 fontsize=13)
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(0, 31, 5))
    plt.tight_layout()
    plt.savefig("plots/ablation_finetune_epochs.png", dpi=150)
    plt.close()
    print("Saved: plots/ablation_finetune_epochs.png")


In [27]:

 #Dynamic vs. Static Quantization

def run_aspect3_quantization_comparison():
    """

     results from quantization.py.
    """
    print("\n" + "=" * 65)
    print(" Dynamic vs. Static Quantization")
    print("=" * 65)


    results = {
        'methods': ['FP32\n(32-bit)', 'Dynamic\n(INT8)', 'Static\n(INT8)'],
        'accuracy': [90.69, 90.67, 90.70],
        'size_mb': [18.43, 18.05, 4.65],
        'latency_ms': [42.61, 35.53, 5.72],
    }

    return results




def plot_aspect3(results):
    os.makedirs("plots", exist_ok=True)

    methods = results['methods']
    colors = ['#2196F3', '#FF9800', '#4CAF50']

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Accuracy
    bars = axes[0].bar(methods, results['accuracy'], color=colors, width=0.5)
    for bar, v in zip(bars, results['accuracy']):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                     f'{v:.2f}%', ha='center', fontsize=10)
    axes[0].set_ylabel('Test Accuracy (%)', fontsize=11)
    axes[0].set_title('Accuracy', fontsize=12)
    axes[0].grid(True, alpha=0.3, axis='y')







    min_acc = min(results['accuracy']) - 0.5
    max_acc = max(results['accuracy']) + 0.5
    axes[0].set_ylim(min_acc, max_acc)






    # Model size
    bars = axes[1].bar(methods, results['size_mb'], color=colors, width=0.5)
    for bar, v in zip(bars, results['size_mb']):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f'{v:.1f} MB', ha='center', fontsize=10)
    axes[1].set_ylabel('Model Size (MB)', fontsize=11)
    axes[1].set_title('Model Size', fontsize=12)
    axes[1].grid(True, alpha=0.3, axis='y')




    # Latency
    bars = axes[2].bar(methods, results['latency_ms'], color=colors, width=0.5)
    for bar, v in zip(bars, results['latency_ms']):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{v:.1f} ms', ha='center', fontsize=10)
    axes[2].set_ylabel('Latency (ms)', fontsize=11)
    axes[2].set_title('Inference Latency', fontsize=12)
    axes[2].grid(True, alpha=0.3, axis='y')

    fig.suptitle('Ablation: Dynamic vs. Static Quantization', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig("plots/ablation_quantization_comparison.png", dpi=150,
                bbox_inches='tight')
    plt.close()
    print("Saved: plots/ablation_quantization_comparison.png")


In [28]:





# Print

def print_all_results(res1_g, res1_l, res2, res3):
    print("\n" + "=" * 75)
    print("STUDY SUMMARY")
    print("=" * 75)

    #1
    print("\n── Global vs. Layer-wise Pruning (after fine-tuning) ──")
    print(f"{'Sparsity':>10} | {'Global':>10} | {'Layer-wise':>12} | {'Difference':>12}")
    print(f"{'-'*50}")
    for g, l in zip(res1_g, res1_l):
        diff = g['acc_after'] - l['acc_after']
        print(f"{g['sparsity']*100:>9.0f}% | {g['acc_after']:>9.2f}% | "
              f"{l['acc_after']:>11.2f}% | {diff:>+11.2f}%")

    # 2
    print(f"\n── Fine-tuning Epochs ──")
    for s in [0.9, 0.95, 0.99]:
        results = res2[s]
        print(f"\n  Sparsity: {s*100:.0f}%")
        print(f"  {'Epochs':>8} | {'Test Acc':>10}")
        print(f"  {'-'*22}")
        for r in results:
            print(f"  {r['epochs']:>8} | {r['acc_after']:>9.2f}%")

    # 3
    print(f"\n── Dynamic vs. Static Quantization ──")
    print(f"{'Method':<15} | {'Accuracy':>9} | {'Size':>8} | {'Latency':>10}")
    print(f"{'-'*50}")
    for i, method in enumerate(res3['methods']):
        m = method.replace('\n', ' ')
        print(f"{m:<15} | {res3['accuracy'][i]:>8.2f}% | "
              f"{res3['size_mb'][i]:>6.2f} MB | "
              f"{res3['latency_ms'][i]:>8.2f} ms")



In [29]:


def plot_combined_figure(res1_g, res1_l, res2, res3):
    """

    """
    os.makedirs("plots", exist_ok=True)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

    # Global vs. Layer-wise ──
    ax = axes[0]
    sparsities = [r['sparsity'] * 100 for r in res1_g]
    acc_g = [r['acc_after'] for r in res1_g]
    acc_l = [r['acc_after'] for r in res1_l]



    ax.plot(sparsities, acc_g, 'bo-', linewidth=2, markersize=7,
            label='Global pruning')
    ax.plot(sparsities, acc_l, 'rs-', linewidth=2, markersize=7,
            label='Layer-wise pruning')



    ax.fill_between(sparsities, acc_l, acc_g, alpha=0.1, color='blue')
    ax.set_xlabel('Sparsity (%)', fontsize=11)
    ax.set_ylabel('Test Accuracy (%)', fontsize=11)
    ax.set_title('(a) Global vs. Layer-wise Pruning', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='lower left')
    ax.grid(True, alpha=0.3)













    #  Fine-tuning Recovery (all sparsities) ──
    ax = axes[1]
    panel_colors = {0.9: '#2196F3', 0.95: '#E91E63', 0.99: '#FF9800'}
    panel_labels = {0.9: '90% sparse', 0.95: '95% sparse', 0.99: '99% sparse'}

    for s in [0.9, 0.95, 0.99]:
        results = res2[s]
        epochs = [r['epochs'] for r in results]
        accs = [r['acc_after'] for r in results]
        acc_before = results[0]['acc_before']
        c = panel_colors[s]

        ax.plot(epochs, accs, 'o-', color=c, linewidth=2, markersize=5,
                label=f'{panel_labels[s]} (start: {acc_before:.1f}%)')
        ax.axhline(y=acc_before, color=c, linestyle='--', linewidth=1, alpha=0.3)
        ax.fill_between(epochs, acc_before, accs, alpha=0.06, color=c)



    ax.axhline(y=90.68, color='gray', linestyle=':', linewidth=1.5,
               label='Unpruned (90.7%)', alpha=0.5)


    ax.set_xlabel('Fine-tuning Epochs', fontsize=11)
    ax.set_ylabel('Test Accuracy (%)', fontsize=11)
    ax.set_title('(b) Fine-tuning Recovery', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)









    #Quantization Comparison ──
    ax = axes[2]
    methods_short = ['FP32', 'Dynamic\nINT8', 'Static\nINT8']
    colors = ['#2196F3', '#FF9800', '#4CAF50']
    latencies = res3['latency_ms']
    accuracies = res3['accuracy']










    # bar chart dude
    bars = ax.bar(methods_short, latencies, color=colors, width=0.5, alpha=0.85)
    for bar, lat, acc in zip(bars, latencies, accuracies):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                f'{lat:.1f} ms\n({acc:.1f}%)', ha='center', fontsize=9)
    ax.set_ylabel('Inference Latency (ms)', fontsize=11)
    ax.set_title('(c) Quantization: Latency & Accuracy', fontsize=12,
                 fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig("plots/ablation_combined.png", dpi=200, bbox_inches='tight')
    plt.close()
    print("Saved: plots/ablation_combined.png")


In [ ]:



# Run everything

if __name__ == "__main__":
    MODEL_PATH = "vgg_cifar10_trained.pth"
    os.makedirs("results", exist_ok=True)

    if not os.path.exists(MODEL_PATH):
        print(f"ERROR: {MODEL_PATH} not found! Run train.py first.")
        exit(1)









    #                     Global vs. Layer-wise
    res1_global, res1_layerwise = run_aspect1_global_vs_layerwise(
        MODEL_PATH, fine_tune_epochs=5
    )
    plot_aspect1(res1_global, res1_layerwise)
    json.dump({'global': res1_global, 'layerwise': res1_layerwise},
              open("results/aspect1_global_vs_layerwise.json", "w"), indent=2)
    print("Saved: results/aspect1_global_vs_layerwise.json")















    # Fine-tuning epochs at 90%, 95%, and 99% sparsity
    res2 = run_aspect2_finetune_epochs(MODEL_PATH)
    plot_aspect2(res2)





    # Convert  for JSON
    res2_json = {str(k): v for k, v in res2.items()}
    json.dump(res2_json, open("results/aspect2_finetune_epochs.json", "w"), indent=2)
    print("Saved: results/aspect2_finetune_epochs.json")














    # Quantization comparison
    res3 = run_aspect3_quantization_comparison()
    plot_aspect3(res3)
    json.dump(res3, open("results/aspect3_quantization.json", "w"), indent=2)
    print("Saved: results/aspect3_quantization.json")


    print_all_results(res1_global, res1_layerwise, res2, res3)

    #
    plot_combined_figure(res1_global, res1_layerwise, res2, res3)

    print("\n\nDone! All ablation plots saved to plots/ directory.")
    print("All data saved to results/ directory.")
    print("Use plots/ablation_combined.png as the main figure in your report.")


Global vs. Layer-wise Pruning

  --- Sparsity: 30% ---
    Global:    90.68% → 90.54% (non-zero: 3,371,962)
    Layerwise: 89.36% → 90.56% (non-zero: 3,371,962)

  --- Sparsity: 50% ---
    Global:    90.68% → 90.62% (non-zero: 2,408,544)
